# D1 · Photovoltaic plant: from the weather forecast to the energy bid (C#)

The same numbers appear in the [PySharp](D1-fotovoltaico-pysharp.ipynb) notebook (`Interop` facade); a test verifies that they coincide. JupyterSharp kernel on .NET 11.

## The case

A **1 MWp photovoltaic plant** (600 kW AC inverter) sells energy on the day-ahead market. The SCADA records irradiance, temperatures, AC power and inverter state every hour; a weather service sends, the day before, the **irradiance forecast** for the next 24 hours, with its error.

**Who works on the data.** The *energy manager* must submit tomorrow's energy bid; whoever does *O&M* wants to know whether the plant performs as it should (and whether washing the modules is worth it); the *owner* wants to know why less was produced on a given day. The customer's question (the trader): **“how much energy do you offer on the market tomorrow?”**, and it must be given **with an uncertainty**, because producing less than offered costs more than producing more.

**Synthetic data** (`generate-data.cs`, with a seed): 8 hourly weeks. The first 7 for training, the last one as a never-seen test. The known truth is in `data/verita.json`: the cleaning of the modules (day 30), an inverter fault (day 38, 5 hours) and the weather service's error.

| Step | Question | Tool |
|---|---|---|
| 1 | Does the plant perform as it should? | KPIs: performance ratio, availability |
| 2 | How much power will it deliver tomorrow, **knowing the forecast weather**? | regression on the covariate, seasonal naive, MASE |
| 3 | With what uncertainty? | quantiles of the residuals by hour, pinball loss, coverage |
| 4 | And on the energy of the **whole day**? | sampled paths (blocks of residuals) |
| 5 | How much energy do I bid? | optimal quantile given the cost of the error |
| 6 | KPIs, dashboard, answer to the customer | tiles, charts, text with the numbers |

In [ ]:
#load "../../../shared/MathemaSharp.PlotSharp.csx"
using MathemaSharp.TimeSeries;
using System.Globalization;
using System.IO;
using System.Runtime.CompilerServices;

static void P(string label, IEnumerable<double> values) =>
    Console.WriteLine($"{label}: {string.Join(" ", values.Select(v => v.ToString("F6", CultureInfo.InvariantCulture)))}");
static void P(string label, double value) => P(label, new[] { value });
static string Here([CallerFilePath] string path = "") => Path.GetDirectoryName(path)!;   // folder of this notebook
static double Num(string s) => double.Parse(s, CultureInfo.InvariantCulture);

var data = Path.GetFullPath(Path.Combine(Here(), "../../../../scenari/D-fotovoltaico/data"));
var rows = File.ReadLines(Path.Combine(data, "impianto_orario.csv")).Skip(1).Select(l => l.Split(',')).ToArray();
double[] G = rows.Select(r => Num(r[1])).ToArray(), Gf = rows.Select(r => Num(r[2])).ToArray(), Tair = rows.Select(r => Num(r[3])).ToArray();
double[] Pac = rows.Select(r => Num(r[5])).ToArray(), ok = rows.Select(r => Num(r[6])).ToArray();
const int trainDays = 49, trainHours = trainDays * 24, total = 56 * 24;
const double pMax = 600, pNom = 1000;   // maximum AC kW of the inverter, installed kWp

## 1 · Acquire and measure: does the plant perform as it should?
The **performance ratio** (IEC 61724) is the energy produced divided by what an ideal plant would produce with the same insolation: $PR=E_{AC}/(H_{POA}\cdot P_{nom})$. It measures the plant's health *regardless of the weather*. The **availability** counts the sunny hours in which the inverter was in service.

In [ ]:
var dailyEnergy = new double[56];
var dailyInsolation = new double[56];
var pr = new double[56];
for (var d = 0; d < 56; d++)
{
    for (var h = 0; h < 24; h++) { dailyEnergy[d] += Pac[d * 24 + h]; dailyInsolation[d] += G[d * 24 + h] / 1000; }
    pr[d] = dailyInsolation[d] > 0.5 ? dailyEnergy[d] / (dailyInsolation[d] * pNom) : double.NaN;
}
double MeanPr(int from, int to) => Enumerable.Range(from, to - from).Select(d => pr[d]).Average();
P("pr_before_cleaning", MeanPr(24, 30));
P("pr_after_cleaning", MeanPr(31, 37));
P("pr_fault_day", pr[38]);
var sunny = Enumerable.Range(0, total).Where(i => G[i] > 50).ToArray();
P("availability", sunny.Count(i => ok[i] == 1) / (double)sunny.Length);
P("hours_at_inverter_limit", Pac.Count(p => p >= pMax - 0.05));

pr_before_cleaning: 0.779849
pr_after_cleaning: 0.808781
pr_fault_day: 0.368400
availability: 0.992636
hours_at_inverter_limit: 17.000000

The PR rises after the cleaning of the modules (day 30) and drops on the day of the fault: the KPI sees both. But the difference due to cleaning is a few points: a **multi-day average** is needed to tell it from the noise.

In [ ]:
Chart.Lines("Daily performance ratio (cleaning on day 30, fault on day 38)", new Line("PR", pr));

Daily performance ratio (cleaning on day 30, fault on day 38) 0.3 0.4 0.5 0.6 0.7 0.8 0.9 0 20 40 60 <path d='M64 81.79L74.53 73.98L85.07 80.08L95.6 76.02L106.13 69.48L116.67 85.91L127.2 77.72L137.73 77.3L148.27 77.51L158.8 85.11L169.33 73.96L179.87 84.58L190.4 80.21L200.93 82.62L211.47 79.93L222 81.88L232.53 79.75L243.07 77L253.6 79.42L264.13 81.59L274.67 81.72L285.2 81.23L295.73 79.08L306.27 83.63L316.8 81.09L327.33 73.16L337.87 81.57L348.4 83.5L358.93 80.75L369.47 82.58L380 72.79L390.53 70L401.07 79.08L411.6 69.4L422.13 67.14L432.67 74.45L443.2 75.7L453.73 72.11L464.27 191.53L474.8 75.57L485.33 73.64L495.87 74.93L506.4 78.08L516.93 75.32L527.47 70.33L538 70.74L548.53 76.09L559.07 78.35L569.6 77.96L580.13 69.18L590.67 78.82L601.2 78.76L611.73 75.33L622.27 78.65L632.8 69.95L643.33 75.34' fill='none' stroke='#2a78d6' stroke-width='2' stroke-linejoin='round' stroke-linecap='round'/> PR · x=0: 0.775 PR · x=1: 0.804 PR · x=2: 0.781 PR · x=3: 0.796 PR · x=4: 0.82 PR · x=5: 0.76 PR · x=6: 0.79 PR · x=7: 0.791 PR · x=8: 0.791 PR · x=9: 0.763 PR · x=10: 0.804 PR · x=11: 0.765 PR · x=12: 0.781 PR · x=13: 0.772 PR · x=14: 0.782 PR · x=15: 0.775 PR · x=16: 0.782 PR · x=17: 0.793 PR · x=18: 0.784 PR · x=19: 0.776 PR · x=20: 0.775 PR · x=21: 0.777 PR · x=22: 0.785 PR · x=23: 0.768 PR · x=24: 0.777 PR · x=25: 0.807 PR · x=26: 0.776 PR · x=27: 0.769 PR · x=28: 0.779 PR · x=29: 0.772 PR · x=30: 0.808 PR · x=31: 0.819 PR · x=32: 0.785 PR · x=33: 0.821 PR · x=34: 0.829 PR · x=35: 0.802 PR · x=36: 0.797 PR · x=37: 0.811 PR · x=38: 0.368 PR · x=39: 0.798 PR · x=40: 0.805 PR · x=41: 0.8 PR · x=42: 0.789 PR · x=43: 0.799 PR · x=44: 0.817 PR · x=45: 0.816 PR · x=46: 0.796 PR · x=47: 0.788 PR · x=48: 0.789 PR · x=49: 0.822 PR · x=50: 0.786 PR · x=51: 0.786 PR · x=52: 0.799 PR · x=53: 0.786 PR · x=54: 0.819 PR · x=55: 0.799

## 2 · Forecast tomorrow's power, knowing the forecast weather
Power depends on irradiance and a weather service forecasts it the day before: this is the **covariate known in the future** (with its error). Model: regression $P\approx\beta_0+\beta_1G_{prev}+\beta_2G_{prev}\,T_{air}$, limited to 0…600 kW, trained by QR on the hours when the inverter was in service. Reference: the **one-day seasonal naive** (the same hour as yesterday).

In [ ]:
var trainRows = Enumerable.Range(0, trainHours).Where(i => ok[i] == 1).ToArray();   // excluding the fault hours: they are not "normal"
var X = new double[trainRows.Length, 3];
var y = new double[trainRows.Length];
for (var r = 0; r < trainRows.Length; r++)
{
    var i = trainRows[r];
    X[r, 0] = 1; X[r, 1] = Gf[i]; X[r, 2] = Gf[i] * Tair[i];
    y[r] = Pac[i];
}
var beta = MathemaSharp.LinearAlgebra.MatrixAlgebra.LeastSquares<double>(X, y);
P("beta", beta);
double Predict(int i) => Gf[i] <= 0 ? 0 : Math.Clamp(beta[0] + beta[1] * Gf[i] + beta[2] * Gf[i] * Tair[i], 0, pMax);

double[] train = Pac[..trainHours], test = Pac[trainHours..];
var model = Enumerable.Range(trainHours, 168).Select(Predict).ToArray();
var persistence = Pac[(trainHours - 24)..(total - 24)];                              // yesterday at the same hour
P("mase_model", Accuracy.Mase<double>(test, model, train, 24));
P("mase_persistence", Accuracy.Mase<double>(test, persistence, train, 24));
try { Accuracy.Mape<double>(test, model); P("mape_defined", 1); } catch (ArgumentException) { P("mape_defined", 0); }   // at night the power is 0

beta: 10.992833 0.806969 -0.002707
mase_model: 0.510989
mase_persistence: 1.361532
mape_defined: 0.000000

**The MAPE cannot be computed**: at night the power is zero and the library says so instead of returning a meaningless number. This is why this book uses the **MASE**: defined even with zeros and scale-independent.

## 3 · With what uncertainty? Quantiles of the residuals by hour of the day
Forecasts err differently at every hour (at noon the error is large, at night nil). The model's **in-sample residuals** (actual − forecast power) are taken **hour by hour** and their quantiles computed: these are the offsets to add to the forecast to obtain P10, P50, P90.

In [ ]:
var residual = new double[trainHours];
for (var i = 0; i < trainHours; i++) residual[i] = Pac[i] - Predict(i);
double[] HourQuantile(int hour, double p) => new[] { SeriesStatistics.Quantile<double>(trainRows.Where(i => i % 24 == hour).Select(i => residual[i]).ToArray(), p) };
var q10 = Enumerable.Range(0, 24).Select(h => HourQuantile(h, 0.1)[0]).ToArray();
var q50 = Enumerable.Range(0, 24).Select(h => HourQuantile(h, 0.5)[0]).ToArray();
var q90 = Enumerable.Range(0, 24).Select(h => HourQuantile(h, 0.9)[0]).ToArray();

double Band(int i, double[] q) => Gf[i] <= 0 ? 0 : Math.Clamp(Predict(i) + q[i % 24], 0, pMax);
var lo = Enumerable.Range(trainHours, 168).Select(i => Band(i, q10)).ToArray();
var mid = Enumerable.Range(trainHours, 168).Select(i => Band(i, q50)).ToArray();
var hi = Enumerable.Range(trainHours, 168).Select(i => Band(i, q90)).ToArray();
P("pinball_p10", Accuracy.PinballLoss<double>(test, lo, 0.1));
P("pinball_p50", Accuracy.PinballLoss<double>(test, mid, 0.5));
P("pinball_p90", Accuracy.PinballLoss<double>(test, hi, 0.9));
P("coverage_p10_p90_all_hours", Accuracy.IntervalCoverage<double>(test, lo, hi));
// At night the power is 0 and the interval is [0, 0]: it always contains the value and inflates the coverage. We measure only on the sunny hours.
var daylight = Enumerable.Range(0, 168).Where(t => Gf[trainHours + t] > 0).ToArray();
double[] At(double[] x) => daylight.Select(t => x[t]).ToArray();
var coverageDay = Accuracy.IntervalCoverage<double>(At(test), At(lo), At(hi));
P("coverage_p10_p90_sun_hours", coverageDay);

pinball_p10: 4.859385
pinball_p50: 11.591131
pinball_p90: 4.671934
coverage_p10_p90_all_hours: 0.916667
coverage_p10_p90_sun_hours: 0.846154

The P10–P90 interval should contain **80 %** of the observations. Counting **all** the hours the coverage looks high, but it is an illusion: at night the power is zero and the interval is [0, 0], so it always contains the value. Measured **on the sunny hours only** it is the number that counts; if it came out lower than 80 %, the interval would be too optimistic (the in-sample residuals underestimate the error on new data).

In [ ]:
int a = 96, b = 168;   // the last 3 days of the test
Chart.Lines("Last 3 days: actual power and P10–P90 band (kW, x axis: hours)",
    new Line("actual", test[a..b]), new Line("P50", mid[a..b], Dashed: true), new Line("P10", lo[a..b], Dashed: true), new Line("P90", hi[a..b], Dashed: true));

Last 3 days: actual power and P10–P90 band (kW, x axis: hours) 0 200 400 600 0 20 40 60 80 <path d='M64 210L71.9 210L79.8 210L87.7 210L95.6 210L103.5 210L111.4 210L119.3 208.66L127.2 178.7L135.1 148.54L143 137.62L150.9 124.06L158.8 95.11L166.7 83.02L174.6 89.5L182.5 102.77L190.4 126.6L198.3 150.46L206.2 182.93L214.1 208.85L222 210L229.9 210L237.8 210L245.7 210L253.6 210L261.5 210L269.4 210L277.3 210L285.2 210L293.1 210L301 210L308.9 209.62L316.8 192.67L324.7 189.67L332.6 188.4L340.5 172.99L348.4 183.96L356.3 175.78L364.2 187.49L372.1 170.18L380 184.42L387.9 182.33L395.8 196.68L403.7 209.69L411.6 210L419.5 210L427.4 210L435.3 210L443.2 210L451.1 210L459 210L466.9 210L474.8 210L482.7 210L490.6 210L498.5 209.83L506.4 183.67L514.3 159.14L522.2 133.34L530.1 132.19L538 125.54L545.9 115.13L553.8 117.62L561.7 138.12L569.6 136.06L577.5 161.59L585.4 187.66L593.3 209.83L601.2 210L609.1 210L617 210L624.9 210' fill='none' stroke='#2a78d6' stroke-width='2' stroke-linejoin='round' stroke-linecap='round'/><path d='M64 210L71.9 210L79.8 210L87.7 210L95.6 210L103.5 210L111.4 210L119.3 207.18L127.2 181.22L135.1 150.26L143 125.13L150.9 102.97L158.8 108.35L166.7 101.37L174.6 98.02L182.5 111.62L190.4 124.45L198.3 152.52L206.2 179.32L214.1 206.92L222 210L229.9 210L237.8 210L245.7 210L253.6 210L261.5 210L269.4 210L277.3 210L285.2 210L293.1 210L301 210L308.9 207.75L316.8 186.74L324.7 166.38L332.6 159.55L340.5 147.96L348.4 154.35L356.3 155.6L364.2 154.59L372.1 159.31L380 170.92L387.9 179.59L395.8 193.14L403.7 207.73L411.6 210L419.5 210L427.4 210L435.3 210L443.2 210L451.1 210L459 210L466.9 210L474.8 210L482.7 210L490.6 210L498.5 208.06L506.4 183.25L514.3 155.27L522.2 131.26L530.1 118.96L538 125.87L545.9 124.06L553.8 118.03L561.7 122.48L569.6 131.97L577.5 162.69L585.4 186.97L593.3 207.91L601.2 210L609.1 210L617 210L624.9 210' fill='none' stroke='#eb6834' stroke-width='2' stroke-dasharray='6 4' stroke-linejoin='round' stroke-linecap='round'/><path d='M64 210L71.9 210L79.8 210L87.7 210L95.6 210L103.5 210L111.4 210L119.3 210L127.2 191.07L135.1 168.45L143 152.56L150.9 132.68L158.8 135.24L166.7 123.8L174.6 118.61L182.5 133.15L190.4 149.29L198.3 165.08L206.2 187.61L214.1 209.95L222 210L229.9 210L237.8 210L245.7 210L253.6 210L261.5 210L269.4 210L277.3 210L285.2 210L293.1 210L301 210L308.9 210L316.8 196.6L324.7 184.58L332.6 186.97L340.5 177.66L348.4 181.25L356.3 178.02L364.2 175.18L372.1 180.84L380 195.76L387.9 192.15L395.8 201.43L403.7 210L411.6 210L419.5 210L427.4 210L435.3 210L443.2 210L451.1 210L459 210L466.9 210L474.8 210L482.7 210L490.6 210L498.5 210L506.4 193.11L514.3 173.47L522.2 158.68L530.1 148.66L538 152.76L545.9 146.48L553.8 138.62L561.7 144.02L569.6 156.81L577.5 175.25L585.4 195.27L593.3 210L601.2 210L609.1 210L617 210L624.9 210' fill='none' stroke='#1baf7a' stroke-width='2' stroke-dasharray='6 4' stroke-linejoin='round' stroke-linecap='round'/><path d='M64 210L71.9 210L79.8 210L87.7 210L95.6 210L103.5 210L111.4 210L119.3 200.64L127.2 171.32L135.1 135.25L143 104.52L150.9 77.43L158.8 80.98L166.7 78.86L174.6 74L182.5 89.58L190.4 105.79L198.3 134.04L206.2 170.48L214.1 202.01L222 210L229.9 210L237.8 210L245.7 210L253.6 210L261.5 210L269.4 210L277.3 210L285.2 210L293.1 210L301 210L308.9 201.2L316.8 176.85L324.7 151.37L332.6 138.93L340.5 122.41L348.4 126.98L356.3 133.08L364.2 130.57L372.1 137.27L380 152.27L387.9 161.1L395.8 184.3L403.7 202.82L411.6 210L419.5 210L427.4 210L435.3 210L443.2 210L451.1 210L459 210L466.9 210L474.8 210L482.7 210L490.6 210L498.5 201.52L506.4 173.35L514.3 140.27L522.2 110.64L530.1 93.41L538 98.49L545.9 101.54L553.8 94.01L561.7 100.44L569.6 113.31L577.5 144.21L585.4 178.13L593.3 203L601.2 210L609.1 210L617 210L624.9 210' fill='none' stroke='#eda100' stroke-width='2' stroke-dasharray='6 4' stroke-linejoin='round' stroke-linecap='round'/> actual · x=0: 0 actual · x=1: 0 actual · x=2: 0 actual · x=3: 0 actual · x=4: 0 actual · x=5: 0 actual · x=6: 0 actual · x=7: 5.6 actual · x=8: 130 actual · x=9: 256 actual · x=10: 302 actual · x=11

## 4 · And the energy of the **whole day**?
The trader's question concerns the kWh of the day, not the power of one hour. **Summing the hourly P10s does not give the day's P10**: it amounts to assuming that all 24 hours err together in the worst possible way. Instead, the **dependencies over time** are preserved by sampling paths: for each training day its entire residual vector is taken, added to the forecast, and one possible energy is obtained. From the 48 possible energies the quantiles are taken (fault day excluded).

In [ ]:
var okDays = Enumerable.Range(0, trainDays).Where(d => d != 38).ToArray();
var p10Day = new double[7]; var p50Day = new double[7]; var p90Day = new double[7]; var p25Day = new double[7];
var actualDay = new double[7]; var sumLo = new double[7]; var sumHi = new double[7];
var trajectoryTotals = new double[7][];
for (var d = 0; d < 7; d++)
{
    var totals = new double[okDays.Length];
    for (var k = 0; k < okDays.Length; k++)
    {
        double s = 0;
        for (var h = 0; h < 24; h++)
        {
            var i = trainHours + d * 24 + h;
            s += Gf[i] <= 0 ? 0 : Math.Clamp(model[d * 24 + h] + residual[okDays[k] * 24 + h], 0, pMax);
        }
        totals[k] = s;
    }
    trajectoryTotals[d] = totals;
    p10Day[d] = SeriesStatistics.Quantile<double>(totals, 0.1);
    p25Day[d] = SeriesStatistics.Quantile<double>(totals, 0.25);
    p50Day[d] = SeriesStatistics.Quantile<double>(totals, 0.5);
    p90Day[d] = SeriesStatistics.Quantile<double>(totals, 0.9);
    for (var h = 0; h < 24; h++) { actualDay[d] += test[d * 24 + h]; sumLo[d] += lo[d * 24 + h]; sumHi[d] += hi[d * 24 + h]; }
}
P("energy_actual", actualDay);
P("energy_p10", p10Day);
P("energy_p50", p50Day);
P("energy_p90", p90Day);
P("energy_sum_p10", sumLo);
P("energy_sum_p90", sumHi);
int Inside(double[] low, double[] high) => Enumerable.Range(0, 7).Count(d => actualDay[d] >= low[d] && actualDay[d] <= high[d]);
P("days_within_paths", Inside(p10Day, p90Day));
P("days_within_hourly_sum", Inside(sumLo, sumHi));
P("paths_width_kwh", Enumerable.Range(0, 7).Average(d => p90Day[d] - p10Day[d]));
P("hourly_sum_width_kwh", Enumerable.Range(0, 7).Average(d => sumHi[d] - sumLo[d]));

energy_actual: 1414.800000 4036.600000 3908.600000 3150.400000 3721.700000 1192.200000 3001.100000
energy_p10: 914.216513 2432.517031 2803.159339 1480.914654 2835.150288 1195.885183 2308.782712
energy_p50: 1748.521896 3269.379526 3639.933614 2315.279025 3670.481569 2027.662215 3139.423999
energy_p90: 2443.001177 3963.858806 4334.412895 3009.758305 4364.960850 2722.141495 3833.903279
energy_sum_p10: 796.324391 2315.913303 2686.467392 1365.298881 2718.824564 1081.509182 2195.316611
energy_sum_p90: 2682.711908 4203.569537 4574.123626 3249.469036 4604.671581 2961.852226 4073.614010
days_within_paths: 4.000000
days_within_hourly_sum: 7.000000
paths_width_kwh: 1528.773012
hourly_sum_width_kwh: 1884.336800

The band **from the paths** is narrower than the one obtained by summing the hourly quantiles (about **1.5 MWh** against **1.9 MWh** on average): it takes into account how much the errors offset each other during the day, while the sum assumes the worst case on all the hours together, which is why it always contains all seven days. The days inside the paths band are **4 out of 7** against the expected 80 %: with only 7 days this is compatible with chance (seeing 4 or fewer out of 7 with probability 0.8 happens about 15 times in 100). **This is not a calibration**: many more test origins are needed.

## 5 · How much energy to bid? The quantile that costs least
On the market, **producing less than offered costs more** than producing more: if the shortfall costs $a$ per kWh and the excess $b$, the bid that minimises the expected cost is the **quantile of level $b/(a+b)$** of the energy distribution: this is the pinball loss of the chapter on probabilistic forecasts. With $a=3,\;b=1$ the level is **0.25**.

In [ ]:
double Cost(double offer, double actual) => 3 * Math.Max(offer - actual, 0) + 1 * Math.Max(actual - offer, 0);   // a = 3 (shortfall), b = 1 (excess)
double WeekCost(double[] offers) => Enumerable.Range(0, 7).Sum(d => Cost(offers[d], actualDay[d]));
P("bid_cost_p10", WeekCost(p10Day));
P("bid_cost_p25", WeekCost(p25Day));
P("bid_cost_p50", WeekCost(p50Day));
P("bid_cost_p90", WeekCost(p90Day));
var bestOffer = new[] { ("p10", p10Day), ("p25", p25Day), ("p50", p50Day), ("p90", p90Day) }.OrderBy(o => WeekCost(o.Item2)).First();
Console.WriteLine($"best_bid: {bestOffer.Item1}");

bid_cost_p10: 6469.515011
bid_cost_p25: 5093.702594
bid_cost_p50: 5844.750598
bid_cost_p90: 13593.441976
best_bid: p25

On the test week the 0.25 level costs less than bidding the median (P50) and much less than the P90, as the theory says. With seven days this is a confirmation, not a proof: the theory holds **on average**, and the cost of one week is noisy.

## 6 · Communicate: KPIs, dashboard and answer to the customer

In [ ]:
var lastDay = 6;
Kpi.Tiles(
    new Tile("Performance ratio", $"{MeanPr(31, 37):F3}", $"after cleaning, against {MeanPr(24, 30):F3} before", "ok"),
    new Tile("Availability", $"{100.0 * sunny.Count(i => ok[i] == 1) / sunny.Length:F1} %", "sunny hours with the inverter in service; 5 fault hours on day 38", "ok"),
    new Tile("Energy tomorrow (P25)", $"{p25Day[lastDay]:F0} kWh", $"P10 {p10Day[lastDay]:F0} – P90 {p90Day[lastDay]:F0} kWh; median {p50Day[lastDay]:F0}", "ok"),
    new Tile("P10–P90 coverage", $"{100 * coverageDay:F0} %", "sunny hours, on the test week (expected 80 %)", coverageDay < 0.7 ? "warn" : "ok"),
    new Tile("Model MASE", $"{Accuracy.Mase<double>(test, model, train, 24):F2}", $"against {Accuracy.Mase<double>(test, persistence, train, 24):F2} for persistence (yesterday = today)", "ok"));

Performance ratio 0.809 after cleaning, against 0.780 before ✓ normal Availability 99.3 % sunny hours with the inverter in service; 5 fault hours on day 38 ✓ normal Energy tomorrow (P25) 2768 kWh P10 2309 – P90 3834 kWh; median 3139 ✓ normal P10–P90 coverage 85 % sunny hours, on the test week (expected 80 %) ✓ normal Model MASE 0.51 against 1.36 for persistence (yesterday = today) ✓ normal

In [ ]:
var answer = $"For 20 September we offer {p25Day[lastDay]:F0} kWh. This is the 25 % level of the expected energy distribution (median {p50Day[lastDay]:F0} kWh, 10–90 % interval from {p10Day[lastDay]:F0} to {p90Day[lastDay]:F0} kWh): " +
             $"the market penalises a shortfall three times as much as an excess, and bidding the {100 * 1.0 / 4:F0} % level is what minimises the expected cost. " +
             $"The forecast is based on the irradiance forecast by the weather service: if the weather changes, we will update the bid before the market closes.";
Kpi.Quote("Answer to the customer: “how much energy do you offer on the market tomorrow?”", answer);

var oem = $"The mean performance ratio rose from {MeanPr(24, 30):F3} to {MeanPr(31, 37):F3} after the modules were cleaned on day 30: the dirt cost about {100 * (MeanPr(31, 37) - MeanPr(24, 30)) / MeanPr(31, 37):F1} % of energy. On the fault day (38) the PR dropped to {pr[38]:F2}: 5 hours of inverter downtime.";
Kpi.Quote("Note for O&M", oem);

Answer to the customer: “how much energy do you offer on the market tomorrow?” For 20 September we offer 2768 kWh. This is the 25 % level of the expected energy distribution (median 3139 kWh, 10–90 % interval from 2309 to 3834 kWh): the market penalises a shortfall three times as much as an excess, and bidding the 25 % level is what minimises the expected cost. The forecast is based on the irradiance forecast by the weather service: if the weather changes, we will update the bid before the market closes.

Note for O&M The mean performance ratio rose from 0.780 to 0.809 after the modules were cleaned on day 30: the dirt cost about 3.6 % of energy. On the fault day (38) the PR dropped to 0.37: 5 hours of inverter downtime.

## What we learned
- **The right KPI removes the weather**: the PR separates the plant's health from the weather.
- **A known covariate improves** the forecast a lot, but it brings its own error with it: the uncertainty of the weather forecast *is* the uncertainty of the bid.
- **No MAPE, yes MASE**: with the nightly zeros the MAPE does not exist.
- **Hourly quantiles do not add up**: for the day's energy, paths are sampled, preserving the dependencies over time.
- **The cost of the error decides the quantile**: bidding the median is not optimal when a shortfall costs more than an excess.
- **One test week is not enough** to calibrate the interval: validation on several origins (*rolling origin*) and conformal prediction are in the book, not yet in the library.